# Java Programming – Assignment 5 (CO5)
## Data Persistence Engineering – Files and Databases

**Name:** J.Subhaashree  
**Register Number:** 192424008  
**Platform:** Google Colab | Java 17 | MySQL



## 1. Install Java, MySQL and JDBC driver

In [1]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-17-jdk mysql-server
!java -version
!javac -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package mysql-client-core-8.0.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../00-mysql-client-core-8.0_8.0.46-0ubuntu0.24.04.5_amd64.deb ...
Unpacking mysql-client-core-8.0 (8.0.46-0ubuntu0.24.04.5) ...
Selecting previously unselected package mysql-client-8.0.
Preparing to unpack .../01-mysql-client-8.0_8.0.46-0ubuntu0.24.04.5_amd64.deb ...
Unpacking mysql-client-8.0 (8.0.46-0ubuntu0.24.04.5) ...
Selecting previously unselected package libaio1t64:amd64.
Preparing to unpack .../02-libaio1t64_0.3.113-6build1.1_amd64.deb ...
Unpacking libaio1t64:amd64 (0.3.113-6build1.1) ...
Selecting previously unselected package libmecab2:amd64.
Preparing to unpack .../

In [2]:
!service mysql start
!mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'Colab@12345';"
!mysql -uroot -p'Colab@12345' -e "CREATE DATABASE IF NOT EXISTS transport_db;"
!wget -q https://repo.maven.apache.org/maven2/com/mysql/mysql-connector-j/9.5.0/mysql-connector-j-9.5.0.jar -O /content/mysql-connector.jar
!mysql -uroot -p'Colab@12345' -e "SHOW DATABASES;"
!ls -lh /content/mysql-connector.jar

 * Starting MySQL database server mysqld        su: warning: cannot change directory to /nonexistent: No such file or directory
[ OK ]
mysql: [Warning] Using a password on the command line interface can be insecure.
mysql: [Warning] Using a password on the command line interface can be insecure.
+--------------------+
| Database           |
+--------------------+
| information_schema |
| mysql              |
| performance_schema |
| sys                |
| transport_db       |
+--------------------+
-rw-r--r-- 1 root root 2.5M Oct 26  2025 /content/mysql-connector.jar


# PART A – Persistence Design & Pseudocode

## A1 – Design a File Format

**Text layout:** `passId,regNo,name,routeNo,boardingStop,validityDate,amountPaid`, one UTF-8 record per line. Use a CSV-aware encoder/parser: quote fields containing commas and escape embedded quotes by doubling them.

**Binary layout using DataOutputStream:** `writeUTF(passId)`, `writeUTF(regNo)`, `writeUTF(name)`, `writeInt(routeNo)`, `writeUTF(boardingStop)`, `writeUTF(validityDate)`, `writeDouble(amountPaid)`. Read in exactly the same order with matching read methods.

**ADD pseudocode:** Open `FileWriter(file,true)` wrapped in `BufferedWriter`; validate fields; encode one CSV record; write record and newline; close writer.

**FIND pseudocode:** Open `FileReader` wrapped in `BufferedReader`; read line-by-line; parse using CSV parser; skip malformed records; return matching pass ID; otherwise return NOT FOUND; close reader.

**LIST BY ROUTE pseudocode:** Read each line using `BufferedReader`; parse; skip malformed lines; display records whose route matches; close reader.

**Damaged data:** Check field count, numeric values and date format. Skip/quarantine malformed records and log line number. Report a missing file clearly; create it on first write. For incomplete writes, write a temporary file and atomically rename for full rewrites; for append, detect truncated final records using a checksum/length marker and retain backups.

**RandomAccessFile alternative:** Fixed-size record i is at offset `i × recordSize`; seeking to a known index is O(1). Finding an arbitrary ID still requires O(n) scan unless an index is maintained. Padding fixed-width fields wastes space.

## A2 – Don't Read What You Don't Need

**A2.1:** 50 MiB = 50 × 1,048,576 = 52,428,800 bytes. A byte-at-a-time `FileInputStream.read()` loop makes about 52.4 million Java read calls, causing repeated method/native I/O overhead.

**A2.2:** Use `BufferedInputStream` or an 8 KiB byte array. The program still examines all bytes (O(n)), but 8 KiB refills reduce the approximate bulk-read operations to 52,428,800 / 8,192 = 6,400.

**A2.3:** Buffering may offer little benefit if the source already supplies large blocks, data is already in memory, or CPU processing dominates.

**Files.readAllLines alternative:** O(file size) memory plus per-line object overhead; buffered streaming uses O(buffer size) extra memory. Streaming is preferable for a 50 MB file under memory constraints; `readAllLines()` is convenient for small files needing all lines in memory.

## A3 – Discover the Data-Access Pattern

**Common steps:** obtain connection, prepare/execute SQL, bind parameters, iterate ResultSet, map columns to objects, handle exceptions and close resources. Differences are table names, SQL, entity fields, keys and row mapping.

**Generic interface:**
```java
public interface Dao<T,K> {
  void insert(T value) throws SQLException;
  Optional<T> findById(K id) throws SQLException;
  List<T> findAll() throws SQLException;
  void update(T value) throws SQLException;
  void delete(K id) throws SQLException;
}
```
`StudentDao implements Dao<Student,Integer>` and contains student-specific SQL and mapping.

**Mapping:** `Student.id → student.student_id (PRIMARY KEY)`; `registerNumber → reg_no (UNIQUE)`; `name → student_name`; `department → department`; `year → study_year`.

**Alternative:** SQL inside GUI/service classes means a renamed column may require edits in many places. DAO centralizes SQL in one implementation. Both have the same underlying database-operation complexity; DAO adds only small abstraction overhead.

## A4 – What Should Be Serialized?

Mark the live JDBC `Connection` field `transient`; it is an external live resource, not a serializable snapshot. Passwords should not be serialized in plaintext; exclude them and use secure credential storage. `Hostel`, `Room` and `Resident` must implement `Serializable`. Serializing Hostel traverses its reachable rooms and residents.

When adding `phoneNumber`, an old file normally gives the new reference field its default value (`null`) if the class remains serialization-compatible. A changed `serialVersionUID` can cause `InvalidClassException`. Declare an explicit UID and migration/defaulting logic.

**CSV/JSON alternative:** Human-readable, portable and easier to evolve, but requires parsing/mapping and escaping. Java serialization handles object graphs conveniently but is Java-specific and version-sensitive. Both save/load n residents in O(n); text formats are often larger but easier to inspect and migrate.

## A5 – Correctness and Safety

**A5.1 transaction pseudocode:** Obtain connection → save old auto-commit → set auto-commit false → read/lock both wallet rows → verify hostel balance ≥ ₹500 → debit hostel ₹500 → credit canteen ₹500 → commit. On any failure, rollback. Restore original auto-commit. Both updates are atomic.

**A5.2 injection:** Concatenating user input lets text become SQL syntax. An ID such as `' OR '1'='1' -- ` can make the condition true and bypass the password check. Safe design:
```java
String sql = "SELECT * FROM users WHERE id=? AND pwd=?";
try (PreparedStatement ps=con.prepareStatement(sql)) {
  ps.setString(1,id);
  ps.setString(2,pwd);
  try (ResultSet rs=ps.executeQuery()) {
    boolean authenticated=rs.next();
  }
}
```
Real systems should store salted password hashes, not plaintext passwords.

**Auto-commit alternative:** If debit commits and the program fails before credit, money is lost. One transaction adds coordination/logging overhead but guarantees atomicity and correctness.

# PART B – Implementation & Experimental Investigation

## B1 – File Store and Buffered I/O

The supplied Complete-the-Code example uses five fields (`passId,regNo,name,route,amount`). The implementation below completes its `find()` TODO, skips damaged lines, tests valid/missing IDs, and measures buffered/unbuffered line counting and search growth.

In [3]:
%%writefile PassStore.java
import java.io.*;
import java.nio.file.*;
import java.util.*;
public class PassStore {
  private final Path file;
  PassStore(Path file){this.file=file;}
  public void add(String id,String reg,String name,int route,double amount)throws IOException{
    try(BufferedWriter w=Files.newBufferedWriter(file,StandardOpenOption.CREATE,StandardOpenOption.APPEND)){
      w.write(String.join(",",id,reg,name,""+route,""+amount)); w.newLine();
    }
  }
  public String[] find(String id)throws IOException{
    if(!Files.exists(file))return null;
    try(BufferedReader r=Files.newBufferedReader(file)){
      String line;
      while((line=r.readLine())!=null){
        String[] f=line.split(",",-1);
        if(f.length!=5)continue;
        try{Integer.parseInt(f[3]);Double.parseDouble(f[4]);}
        catch(NumberFormatException e){continue;}
        if(f[0].equals(id))return f;
      }
    }
    return null;
  }
  public static long unbuffered(Path p)throws IOException{
    long n=0;try(InputStream in=Files.newInputStream(p)){int b;while((b=in.read())!=-1)if(b==10)n++;}return n;
  }
  public static long buffered(Path p)throws IOException{
    long n=0;try(InputStream in=new BufferedInputStream(Files.newInputStream(p),8192)){int b;while((b=in.read())!=-1)if(b==10)n++;}return n;
  }
  public static void main(String[] a)throws Exception{
    Path f=Paths.get("passes.csv");Files.deleteIfExists(f);PassStore s=new PassStore(f);
    s.add("P1001","192311001","Arun",12,4500);
    Files.writeString(f,"P1002,DAMAGED LINE\n",StandardOpenOption.APPEND);
    s.add("P1003","192311003","Divya",7,3800);
    System.out.println("Existing: "+Arrays.toString(s.find("P1003")));
    System.out.println("Damaged ID: "+Arrays.toString(s.find("P1002")));
    System.out.println("Missing ID: "+Arrays.toString(s.find("P9999")));
    System.out.println("Damaged row skipped; scan continues.");
  }
}

Writing PassStore.java


In [4]:
!javac PassStore.java
!java PassStore

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Existing: [P1003, 192311003, Divya, 7, 3800.0]
Damaged ID: null
Missing ID: null
Damaged row skipped; scan continues.


### B1.3 – Measure buffering on 1 MB, 10 MB and 50 MB

In [5]:
%%writefile BufferBenchmark.java
import java.nio.file.*;import java.util.*;
public class BufferBenchmark{
 public static void main(String[]x)throws Exception{
  for(int mb:new int[]{1,10,50}){
   Path p=Paths.get("log_"+mb+"MB.txt");byte[] b=new byte[1024*1024];Arrays.fill(b,(byte)'x');b[b.length-1]=10;
   try(var o=Files.newOutputStream(p)){for(int i=0;i<mb;i++)o.write(b);}
   long t=System.nanoTime();long a=PassStore.unbuffered(p);long u=System.nanoTime();
   long z=PassStore.buffered(p);long v=System.nanoTime();
   System.out.printf("%d MB lines=%d unbuffered=%.3f ms buffered=%.3f ms%n",mb,a,(u-t)/1e6,(v-u)/1e6);
  }
 }
}

Writing BufferBenchmark.java


In [6]:
!javac BufferBenchmark.java
!java BufferBenchmark

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
1 MB lines=1 unbuffered=891.958 ms buffered=45.980 ms
10 MB lines=10 unbuffered=8831.046 ms buffered=303.326 ms
50 MB lines=50 unbuffered=39576.871 ms buffered=1294.046 ms


### B1.4 – Measure search growth

In [7]:
%%writefile GrowthBenchmark.java
import java.nio.file.*;
public class GrowthBenchmark{
 public static void main(String[]x)throws Exception{
  for(int n:new int[]{1000,10000,100000}){
   Path p=Paths.get("growth_"+n+".csv");
   try(var w=Files.newBufferedWriter(p)){for(int i=0;i<n;i++)w.write("P"+i+",192424008,Student"+i+",12,4500.0\n");}
   PassStore s=new PassStore(p);long t=System.nanoTime();boolean found=s.find("P"+(n-1))!=null;long e=System.nanoTime();
   System.out.printf("records=%d found=%s time=%.3f ms%n",n,found,(e-t)/1e6);
  }
 }
}

Writing GrowthBenchmark.java


In [8]:
!javac GrowthBenchmark.java
!java GrowthBenchmark

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
records=1000 found=true time=57.699 ms
records=10000 found=true time=106.557 ms
records=100000 found=true time=238.347 ms


**B1 test interpretation:** Valid ID returns fields; missing ID returns null; malformed line is skipped without stopping later searches. The design uses buffered line-by-line reading. The five-field starter format assumes names contain no commas; production code should use a CSV-aware parser.

## B2 – Adversarial Input

The next cell tests an unsafe concatenated login query against a parameterized PreparedStatement in a disposable local table.

In [9]:
%%writefile LoginSecurity.java
import java.sql.*;
public class LoginSecurity{
 public static void main(String[]x)throws Exception{
  String url="jdbc:mysql://localhost:3306/transport_db";
  try(Connection c=DriverManager.getConnection(url,"root","Colab@12345");Statement s=c.createStatement()){
   s.executeUpdate("DROP TABLE IF EXISTS users_test");
   s.executeUpdate("CREATE TABLE users_test(id VARCHAR(50) PRIMARY KEY,pwd VARCHAR(100))");
   s.executeUpdate("INSERT INTO users_test VALUES('student1','correct123')");
   String id="' OR '1'='1' -- ",pwd="wrong";
   String unsafe="SELECT * FROM users_test WHERE id='"+id+"' AND pwd='"+pwd+"'";
   try(ResultSet r=s.executeQuery(unsafe)){System.out.println("Unsafe accepted? "+r.next());}
   try(PreparedStatement p=c.prepareStatement("SELECT * FROM users_test WHERE id=? AND pwd=?")){
    p.setString(1,id);p.setString(2,pwd);
    try(ResultSet r=p.executeQuery()){System.out.println("PreparedStatement accepted? "+r.next());}
   }
  }
 }
}

Writing LoginSecurity.java


In [10]:
!javac -cp /content/mysql-connector.jar LoginSecurity.java
!java -cp .:/content/mysql-connector.jar LoginSecurity

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Unsafe accepted? true
PreparedStatement accepted? false


**Damaged input tests:** Wrong field count, empty lines and non-numeric route/amount values are skipped by `PassStore.find()`. This prevents a single malformed record from stopping the scan.

**Alternative analysis:** Escaping quotes or regex validation is weaker because SQL syntax/encoding edge cases are difficult to handle completely. PreparedStatement separates SQL structure from values. Validation remains useful for business rules, but is not a substitute for parameterized SQL.

## B3 – Statement vs PreparedStatement vs Batch

This program compares all three approaches at 10, 1,000, 10,000 and 100,000 rows, checks ResultSetMetaData, and demonstrates rollback after a deliberately forced mid-load failure.

In [11]:
%%writefile JdbcBenchmarks.java
import java.sql.*;
public class JdbcBenchmarks{
 static String url="jdbc:mysql://localhost:3306/transport_db",u="root",pw="Colab@12345";
 static String ins="INSERT INTO bus_pass VALUES(?,?,?,?,?)";
 static void reset(Connection c)throws Exception{try(Statement s=c.createStatement()){s.executeUpdate("DROP TABLE IF EXISTS bus_pass");s.executeUpdate("CREATE TABLE bus_pass(pass_id VARCHAR(30) PRIMARY KEY,reg_no VARCHAR(30),name VARCHAR(80),route_no INT,amount DOUBLE)");}}
 static long count(Connection c)throws Exception{try(Statement s=c.createStatement();ResultSet r=s.executeQuery("SELECT COUNT(*) FROM bus_pass")){r.next();return r.getLong(1);}}
 static void statement(Connection c,int n)throws Exception{try(Statement s=c.createStatement()){for(int i=0;i<n;i++)s.executeUpdate("INSERT INTO bus_pass VALUES('S"+i+"','192424008','Student"+i+"',12,4500.0)");}}
 static void prepared(Connection c,int n)throws Exception{try(PreparedStatement p=c.prepareStatement(ins)){for(int i=0;i<n;i++){p.setString(1,"P"+i);p.setString(2,"192424008");p.setString(3,"Student"+i);p.setInt(4,12);p.setDouble(5,4500);p.executeUpdate();}}}
 static void batch(Connection c,int n)throws Exception{boolean old=c.getAutoCommit();c.setAutoCommit(false);try(PreparedStatement p=c.prepareStatement(ins)){for(int i=0;i<n;i++){p.setString(1,"B"+i);p.setString(2,"192424008");p.setString(3,"Student"+i);p.setInt(4,12);p.setDouble(5,4500);p.addBatch();if((i+1)%1000==0)p.executeBatch();}p.executeBatch();c.commit();}catch(Exception e){c.rollback();throw e;}finally{c.setAutoCommit(old);}}
 public static void main(String[]z)throws Exception{
  try(Connection c=DriverManager.getConnection(url,u,pw)){
   for(int n:new int[]{10,1000,10000,100000}){
    reset(c);long t=System.nanoTime();statement(c,n);System.out.printf("Statement %d rows: %.3f ms%n",n,(System.nanoTime()-t)/1e6);
    if(n==10)try(Statement s=c.createStatement();ResultSet r=s.executeQuery("SELECT * FROM bus_pass");){ResultSetMetaData m=r.getMetaData();System.out.println("Metadata columns="+m.getColumnCount()+" first="+m.getColumnName(1)+" type="+m.getColumnTypeName(1));}
    reset(c);t=System.nanoTime();prepared(c,n);System.out.printf("PreparedStatement %d rows: %.3f ms%n",n,(System.nanoTime()-t)/1e6);
    reset(c);t=System.nanoTime();batch(c,n);System.out.printf("Batch transaction %d rows: %.3f ms%n",n,(System.nanoTime()-t)/1e6);
   }
   reset(c);c.setAutoCommit(false);
   try(PreparedStatement p=c.prepareStatement(ins)){
    for(int i=0;i<10;i++){if(i==5)throw new SQLException("Forced failure halfway");p.setString(1,"R"+i);p.setString(2,"192424008");p.setString(3,"Rollback"+i);p.setInt(4,12);p.setDouble(5,4500);p.addBatch();}
    p.executeBatch();c.commit();
   }catch(Exception e){c.rollback();System.out.println("Forced failure: "+e.getMessage());}
   c.setAutoCommit(true);System.out.println("Rows remaining after rollback="+count(c));
  }
 }
}

Writing JdbcBenchmarks.java


In [12]:
!javac -cp /content/mysql-connector.jar JdbcBenchmarks.java
!java -cp .:/content/mysql-connector.jar JdbcBenchmarks

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Statement 10 rows: 142.296 ms
Metadata columns=5 first=pass_id type=VARCHAR
PreparedStatement 10 rows: 145.781 ms
Batch transaction 10 rows: 40.388 ms
Statement 1000 rows: 11402.135 ms
PreparedStatement 1000 rows: 10936.767 ms
Batch transaction 1000 rows: 499.540 ms
Statement 10000 rows: 100108.332 ms
PreparedStatement 10000 rows: 99137.516

**B3 interpretation:** Statement builds SQL strings and is unsafe with untrusted concatenated input. PreparedStatement parameterizes values and can reuse a compiled statement. Batch reduces round trips and is generally suitable for bulk imports. Use PreparedStatement for a single counter insert and batched PreparedStatement in a transaction for a nightly 100,000-row import. The forced failure is rolled back; final row count should be zero.

## B4 – Migration and Database Metadata

This migration reads the five-field pass file, reports invalid rows, inserts valid records in batches in one transaction, and prints database/driver metadata and table names.

In [13]:
%%writefile MigrationAndMetadata.java
import java.io.*;import java.nio.file.*;import java.sql.*;
public class MigrationAndMetadata{
 public static void main(String[]x)throws Exception{
  String url="jdbc:mysql://localhost:3306/transport_db";
  Path f=Paths.get("migration_passes.csv");
  Files.writeString(f,"M1001,192424008,Asha,4,3200.0\nM1002,192424009,Ravi,7,3800.0\nBROKEN,LINE\nM1004,192424010,Meena,not_a_route,4000.0\n");
  try(Connection c=DriverManager.getConnection(url,"root","Colab@12345")){
   try(Statement s=c.createStatement()){s.executeUpdate("DROP TABLE IF EXISTS bus_pass");s.executeUpdate("CREATE TABLE bus_pass(pass_id VARCHAR(30) PRIMARY KEY,reg_no VARCHAR(30),name VARCHAR(80),route_no INT,amount DOUBLE)");}
   boolean old=c.getAutoCommit();c.setAutoCommit(false);int loaded=0,lineNo=0;
   try(BufferedReader r=Files.newBufferedReader(f);PreparedStatement p=c.prepareStatement("INSERT INTO bus_pass VALUES(?,?,?,?,?)")){
    String line;while((line=r.readLine())!=null){lineNo++;String[] a=line.split(",",-1);
     try{if(a.length!=5)throw new IllegalArgumentException("expected 5 fields");int route=Integer.parseInt(a[3]);double amount=Double.parseDouble(a[4]);if(amount<0)throw new IllegalArgumentException("negative amount");p.setString(1,a[0]);p.setString(2,a[1]);p.setString(3,a[2]);p.setInt(4,route);p.setDouble(5,amount);p.addBatch();loaded++;if(loaded%1000==0)p.executeBatch();}
     catch(Exception e){System.out.println("Rejected line "+lineNo+": "+line+" | "+e.getMessage());}
    }
    p.executeBatch();c.commit();
   }catch(Exception e){c.rollback();throw e;}finally{c.setAutoCommit(old);}
   System.out.println("Valid records loaded="+loaded);
   DatabaseMetaData m=c.getMetaData();System.out.println("Database="+m.getDatabaseProductName()+" "+m.getDatabaseProductVersion());System.out.println("Driver="+m.getDriverName()+" "+m.getDriverVersion());
   try(ResultSet t=m.getTables(c.getCatalog(),null,"%",new String[]{"TABLE"})){System.out.println("Tables:");while(t.next())System.out.println(t.getString("TABLE_NAME"));}
  }
 }
}

Writing MigrationAndMetadata.java


In [14]:
!javac -cp /content/mysql-connector.jar MigrationAndMetadata.java
!java -cp .:/content/mysql-connector.jar MigrationAndMetadata

[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Rejected line 3: BROKEN,LINE | expected 5 fields
Rejected line 4: M1004,192424010,Meena,not_a_route,4000.0 | For input string: "not_a_route"
Valid records loaded=2
Database=MySQL 8.0.46-0ubuntu0.24.04.5
Driver=MySQL Connector/J mysql-connector-j-9.5.0 (Revision: a7b3c94f50efbddb9f0dd69b3e0d1aaa25305cd6)
Tables:
bus_pass
users_test


### B4.3 Production decision

For 12 campuses, 100× the current pass volume and concurrent counters, use a relational database. It supports indexed searching, concurrent clients, transactions, constraints, access control and centralized backups. Text files are useful for exports/imports but are weak as a shared concurrent store. Java serialization is Java-specific, difficult to query and hard to coordinate among writers.

If each campus kept its own serialized file, searching all campuses requires checking 12 files (a full scan is roughly O(12n)); concurrent counters need file locking and conflict handling. A shared database is more scalable and safer.

